## **Reto entre clases: la API de un LLM a fondo**

Continuación del laboratorio de clase. Aquí vas un paso más allá: qué devuelve exactamente la API, cómo se controla un modelo
que **razona**, cómo recibir la respuesta en directo, cómo obtener **datos estructurados** fiables y cómo **evaluar** lo que responde.

Modelo: **GPT-6 Luna** con la **Responses API**.

### 🧭 Antes de empezar

- **Cómo se usa:** ejecuta las celdas en orden, de arriba abajo, con `Mayús + Enter`. Si una celda da error, lee la última línea: casi siempre dice qué falta.
- **Si algo se queda raro:** menú *Entorno de ejecución → Reiniciar y ejecutar todo*.
- **Tu API key:** en Colab, guárdala como secreto `OPENAI_API_KEY` (icono de la llave, a la izquierda). Nunca la pegues dentro de una celda.

**Vocabulario de este notebook**

| Término | Qué es |
|---|---|
| **Token** | el trozo de texto con el que trabaja el modelo. Se paga por token, también por los de razonamiento. |
| **`reasoning.effort`** | cuánto piensa el modelo antes de responder (`none`, `low`, `medium`…). |
| **Streaming** | recibir la respuesta mientras se genera, en vez de esperar a que termine. |
| **Salida estructurada** | obligar al modelo a responder con un formato fijo (un modelo de Pydantic) en vez de texto libre. |
| **LLM-as-judge** | usar otro modelo, con una rúbrica, para puntuar las respuestas. |


### Qué practicarás

| Sección | Tema | Clase |
|---|---|---|
| 1-2 | Configuración y anatomía de una respuesta | 1 |
| 3 | Parámetros en modelos que razonan: `effort`, `verbosity`, `max_output_tokens` | 1 |
| 4 | Streaming | 1 |
| 5 | Prompt engineering: instrucciones, few-shot | 1 |
| 6 | Salida estructurada con Pydantic | 2 |
| 7 | Fallos típicos: alucinaciones y sesgo | 2 |
| 8 | Evaluación: coincidencia exacta y LLM-as-judge | 2 |
| **Retos** | Tres ejercicios para completar (la solución está en *Retos_Solucionados*) | |

## Sección 1: Configuración

In [ ]:
!pip install -q --upgrade openai pandas pydantic

In [ ]:
import os
from getpass import getpass

# La clave NUNCA se escribe en el notebook. Orden: secretos de Colab -> variable de entorno -> teclado
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
print("Clave cargada:", os.environ["OPENAI_API_KEY"][:7] + "…")

La clave identifica tu cuenta y se factura contra ella. **Nunca** la escribas en el notebook: en Colab, guárdala en *Secretos* (icono de la llave) con el nombre `OPENAI_API_KEY`; en tu ordenador, como variable de entorno. Si no la encuentra, te la pide sin mostrarla.

In [ ]:
from openai import OpenAI

client = OpenAI()          # lee OPENAI_API_KEY del entorno
MODELO = "gpt-6-luna"      # el GPT-6 más rápido y barato (0,10 $ / 0,50 $ por millón de tokens)
ESFUERZO = "low"           # razonamiento: none | low | medium | high | xhigh | max

def preguntar(pregunta, esfuerzo=ESFUERZO, **kw):
    """Una llamada y devuelve el texto. kw permite pasar instructions, max_output_tokens, text..."""
    r = client.responses.create(model=MODELO, input=pregunta, reasoning={"effort": esfuerzo}, **kw)
    return r.output_text.strip()

import re, unicodedata

def normalizar(t):
    """Minúsculas, sin tildes ni puntuación final: para comparar respuestas cortas («Berlín.» == «berlin»)."""
    t = unicodedata.normalize("NFKD", t.strip().lower()).encode("ascii", "ignore").decode()
    return re.sub(r"[.,;:!?¿¡\s]+$", "", t)

print(preguntar("Di hola en 5 palabras."))

## Sección 2: Anatomía de una respuesta

In [ ]:
r = client.responses.create(
    model=MODELO,
    instructions="Responde en español, en 3 viñetas.",
    input="¿Qué es un LLM y por qué a veces se equivoca contando letras?",
    reasoning={"effort": "medium"},
)
print("TEXTO:\n", r.output_text, "\n")
print("ESTADO:", r.status)
print("PIEZAS DE LA SALIDA:", [item.type for item in r.output])   # 'reasoning' + 'message'
u = r.usage
print(f"TOKENS → entrada: {u.input_tokens} · salida: {u.output_tokens} "
      f"(de ellos razonamiento: {u.output_tokens_details.reasoning_tokens})")

- `output_text`: el texto final, ya unido.
- `output`: la lista de **items** que ha producido el modelo. Si el modelo ha pensado verás un item `reasoning` (el «pensamiento», que no se muestra) además del `message` (la respuesta).
- `usage`: los tokens que pagas. **Los de razonamiento se cobran como salida aunque no los veas.**

## Sección 3: Parámetros en modelos que razonan

Con GPT-6 los mandos principales ya no son `temperature` y `top_p`, sino **cuánto piensa** (`reasoning.effort`) y **cuánto escribe** (`text.verbosity`).
Veámoslo con una tarea que a los LLMs se les da mal: **contar letras** (recuerda: ven tokens, no letras).

In [ ]:
import time

acertijo = "¿Cuántas veces aparece la letra 'e' en la frase 'el elefante tenebroso se entretiene'? Responde solo con el número."

for esfuerzo in ["none", "low", "medium"]:
    t0 = time.time()
    r = client.responses.create(model=MODELO, input=acertijo, reasoning={"effort": esfuerzo})
    razon = r.usage.output_tokens_details.reasoning_tokens
    print(f"effort={esfuerzo:<7} → {r.output_text.strip():<10} | {time.time()-t0:4.1f} s | tokens de razonamiento: {razon}")

La respuesta correcta es **11**. Sin pensar suele fallar; con más esfuerzo acierta. Más esfuerzo = más tokens de razonamiento = más tiempo y coste. La regla: empieza bajo y sube solo si falla.

In [ ]:
for v in ["low", "high"]:
    t = preguntar("Explica qué es el overfitting.", text={"verbosity": v})
    print(f"── verbosity={v}: {len(t)} caracteres\n{t[:300]}…\n")

`verbosity` controla la **longitud** sin tener que pedirlo en el prompt.

In [ ]:
r = client.responses.create(model=MODELO, input="Resume la historia de internet en un tuit.",
                            reasoning={"effort": "medium"}, max_output_tokens=40)
print("estado:", r.status, "| motivo:", r.incomplete_details)
print("texto:", repr(r.output_text))

**Trampa clásica:** `max_output_tokens` **incluye los tokens de razonamiento**. Con un límite pequeño y esfuerzo medio,
el modelo se gasta el presupuesto pensando y la respuesta sale **vacía** (`status='incomplete'`). Deja margen o baja el esfuerzo.

In [ ]:
from openai import BadRequestError

try:
    preguntar("Inventa un nombre para una startup de IA.", esfuerzo="low", temperature=1.3)
except BadRequestError as e:
    print("Error 400 →", e.body.get("message", e)[:120] if isinstance(e.body, dict) else e)

print("Con effort='none' sí se admite:", preguntar("Inventa un nombre para una startup de IA.", esfuerzo="none", temperature=1.3))

`temperature` solo se puede usar **sin razonamiento** (`effort='none'`). Para variedad creativa: `none` + temperatura alta, o pedir varias opciones en el prompt.

## Sección 4: Streaming

In [ ]:
texto = []
with client.responses.stream(model=MODELO, input="Escribe un poema de 4 versos sobre el mar.",
                             reasoning={"effort": "low"}) as stream:
    for evento in stream:
        if evento.type == "response.output_text.delta":
            print(evento.delta, end="", flush=True)
            texto.append(evento.delta)
    final = stream.get_final_response()

print(f"\n\n{len(texto)} fragmentos · {final.usage.output_tokens} tokens de salida")

El **streaming** no abarata nada: mejora la **experiencia**, porque el usuario ve el texto mientras se genera (como en ChatGPT).
Los eventos tienen un `type`; el que trae texto es `response.output_text.delta`. Al final, `get_final_response()` da la respuesta completa con su `usage`.

## Sección 5: Prompt engineering

In [ ]:
print(preguntar("¿Cómo leo un CSV en Python?",
                instructions="Eres un profesor de Python. Responde SIEMPRE con una lista numerada de máximo 3 pasos y un ejemplo de código."))

`instructions` es el *system prompt* de la Responses API: define rol, formato y límites para toda la llamada.

In [ ]:
# Few-shot: los ejemplos van como turnos previos de la conversación
mensajes = [
    {"role": "developer", "content": "Clasifica reseñas como positivo, negativo o neutro. Responde con una sola palabra."},
    {"role": "user", "content": "Superó mis expectativas, lo recomiendo."},      {"role": "assistant", "content": "positivo"},
    {"role": "user", "content": "Llegó roto y nadie me contesta."},              {"role": "assistant", "content": "negativo"},
    {"role": "user", "content": "Es tal cual la descripción, llegó en plazo."},  {"role": "assistant", "content": "neutro"},
    {"role": "user", "content": "Funciona, aunque la batería dura menos de lo prometido."},
]
r = client.responses.create(model=MODELO, input=mensajes, reasoning={"effort": "none"})
print(r.output_text)

**Few-shot**: enseñas el formato con ejemplos en vez de explicarlo. El rol `developer` es el equivalente moderno de `system`.

> ¿Y el *chain-of-thought* («razona paso a paso»)? Con modelos que razonan ya no hace falta pedirlo: se controla con `reasoning.effort`
> (lo has visto en la sección 3). Pedirlo en el prompt solo alarga la respuesta visible.

---
# A partir de aquí: clase 2 (LLMs en producción)
---
## Sección 6: Salida estructurada con Pydantic
Pedir «devuélveme un JSON» y limpiarlo con expresiones regulares es frágil. Con **Structured Outputs** defines el esquema
con Pydantic y la API **garantiza** que la respuesta lo cumple.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Analisis(BaseModel):
    sentimiento: Literal["positivo", "negativo", "neutro"]
    confianza: float = Field(ge=0, le=1)
    palabras_clave: list[str] = Field(description="exactamente 3 palabras clave")

r = client.responses.parse(model=MODELO, input="El servicio fue horrible, esperé 2 horas y la comida llegó fría.",
                           text_format=Analisis, reasoning={"effort": "low"})
a = r.output_parsed          # ¡ya es un objeto Python validado!
print(type(a).__name__, "→", a)
print("Sentimiento:", a.sentimiento, "| confianza:", a.confianza)

In [ ]:
import pandas as pd

resenas = [
    "La pizza estaba deliciosa y el ambiente era muy acogedor. Volveré.",
    "Esperé 45 minutos para que me atendieran y la cuenta tenía errores.",
    "Bien ubicado. La comida es normal, ni destacable ni mala.",
    "El camarero fue encantador pero el postre llegó congelado.",
]
filas = []
for texto in resenas:
    a = client.responses.parse(model=MODELO, input=texto, text_format=Analisis, reasoning={"effort": "none"}).output_parsed
    filas.append({"reseña": texto[:45] + "…", **a.model_dump()})

df = pd.DataFrame(filas)
df

Este patrón (texto libre → objeto validado → DataFrame) es la base de muchos pipelines de datos con LLMs.

## Sección 7: Fallos típicos

In [ ]:
for p in ["¿Qué dijo Napoleón en su famoso discurso de 1802 en Madrid?",
          "Cita textualmente el artículo 4 del Reglamento Europeo de IA."]:
    print(f"P: {p}\nR: {preguntar(p)[:400]}\n")

**Alucinaciones:** la primera pregunta tiene una **premisa falsa** (Napoleón no estuvo en Madrid en 1802). Los modelos actuales
la detectan mucho mejor que hace dos años, pero con citas textuales, cifras exactas o referencias bibliográficas todavía pueden inventar.
Para datos críticos: RAG con citas y verificación.

In [ ]:
tema = "el uso de redes sociales en adolescentes"
print("SESGADO:", preguntar(f"Dame 3 razones por las que {tema} es perjudicial.", text={"verbosity": "low"}), "\n")
print("NEUTRO :", preguntar(f"Dame 2 aspectos positivos y 2 negativos de {tema}.", text={"verbosity": "low"}))

**Sesgo por encuadre:** el modelo sigue el marco de la pregunta. Un prompt con la conclusión dentro produce una respuesta parcial.

## Sección 8: Evaluación

In [ ]:
# normalizar() está definida en la sección 1
benchmark = [
    ("¿Capital de Alemania? Solo el nombre.", "Berlín"),
    ("¿Cuánto es 7 x 8? Solo el número.", "56"),
    ("¿Quién escribió el Quijote? Solo nombre y apellido.", "Miguel de Cervantes"),
    ("¿Símbolo químico del agua? Solo la fórmula.", "H2O"),
    ("¿Año de la llegada a la Luna? Solo el año.", "1969"),
]
filas = []
for pregunta, esperado in benchmark:
    obtenido = preguntar(pregunta, esfuerzo="none")
    filas.append({"pregunta": pregunta, "esperado": esperado, "obtenido": obtenido,
                  "ok": normalizar(obtenido) == normalizar(esperado)})
df_bench = pd.DataFrame(filas)
print(df_bench.to_string(index=False))
print(f"\nAcierto: {df_bench.ok.mean():.0%}")

La **coincidencia exacta** es objetiva y barata, pero solo sirve para respuestas cortas. Normalizar (minúsculas, tildes, puntuación) evita falsos fallos.

In [ ]:
class Juicio(BaseModel):
    puntuacion: int = Field(ge=1, le=5)
    justificacion: str

def juez(pregunta, respuesta):
    rubrica = ("Evalúa la respuesta de 1 a 5. 5 = correcta, completa y clara; 3 = correcta pero incompleta; "
               "1 = incorrecta o vacía. Sé estricto y justifica en una frase.")
    return client.responses.parse(model=MODELO, instructions=rubrica,
                                  input=f"PREGUNTA: {pregunta}\nRESPUESTA: {respuesta}",
                                  text_format=Juicio, reasoning={"effort": "low"}).output_parsed

casos = [("¿Qué es el machine learning?", "Rama de la IA en la que los modelos aprenden patrones a partir de datos para predecir o decidir sin reglas programadas a mano."),
         ("¿Qué es el machine learning?", "Es algo de ordenadores."),
         ("¿Qué es el overfitting?", "Cuando el modelo memoriza el entrenamiento y generaliza mal a datos nuevos.")]
for p, resp in casos:
    j = juez(p, resp)
    print(f"{j.puntuacion}/5  {resp[:60]:<62} → {j.justificacion}")

**LLM-as-judge** con salida estructurada: una **rúbrica explícita** y la puntuación como entero validado.
Recuerda sus sesgos (posición, longitud, autopreferencia): calíbralo con unos cuantos casos etiquetados por ti. Lo vemos a fondo en el **Lab de Evals** de la clase 2.

---
# Retos
Completa las celdas marcadas con `TODO`. Comprueba tu solución ejecutando la celda de verificación de cada reto.

| Reto | Cuándo hacerlo |
|---|---|
| 1 · El esfuerzo mínimo | Después de la clase 1 (secciones 1-5) |
| 2 · Extraer facturas | Después de la clase 2: usa la salida estructurada (sección 6) |
| 3 · Calibra al juez | Después de la clase 2: usa el LLM-as-judge (sección 8) |

### Reto 1 — El esfuerzo mínimo
Encuentra el **nivel de razonamiento más bajo** que resuelve bien los 4 problemas. Así se elige el modelo en producción: lo más barato que funciona.

In [ ]:
problemas = [
    ("Si 5 máquinas hacen 5 piezas en 5 minutos, ¿cuántos minutos tardan 100 máquinas en hacer 100 piezas? Solo el número.", "5"),
    ("Un nenúfar duplica su tamaño cada día y cubre el lago en 48 días. ¿En qué día cubre la mitad? Solo el número.", "47"),
    ("¿Cuántas veces aparece la letra r en 'ferrocarril'? Solo el número.", "4"),
    ("Tengo 3 hermanas y cada una tiene 2 hermanos. Soy chico. ¿Cuántos chicos hay en la familia? Solo el número.", "2"),
]

def aciertos(esfuerzo):
    return sum(normalizar(preguntar(p, esfuerzo=esfuerzo)) == normalizar(esperado) for p, esperado in problemas)

for esfuerzo in ["none", "low", "medium"]:
    n = aciertos(esfuerzo)
    print(f"effort={esfuerzo:<7} → {n}/{len(problemas)}")
    if n == len(problemas):
        print(f"\n✅ Esfuerzo mínimo que lo resuelve todo: {esfuerzo}")
        break

### Reto 2 — Extraer facturas a un DataFrame
Define un esquema Pydantic `Factura` con: `proveedor` (str), `fecha` (str, formato AAAA-MM-DD), `importe_total` (float) y
`categoria` (una de: "software", "viajes", "material", "otros"). Extrae las tres facturas a un DataFrame.

In [ ]:
textos_factura = [
    "Factura n.º 2026-118 de Nubia Cloud SL, emitida el 3 de octubre de 2026. Licencias anuales de software analítico: 1.452,00 € IVA incluido.",
    "RENFE Viajeros. Billete Madrid–Valencia, 21/10/2026. Total pagado 47,30 euros.",
    "Papelería Central · 12-11-2026 · 3 cuadernos, 2 cajas de rotuladores. TOTAL: 23,85€",
]

class Factura(BaseModel):
    proveedor: str
    fecha: str = Field(description="formato AAAA-MM-DD")
    importe_total: float
    categoria: Literal["software", "viajes", "material", "otros"]

facturas = []
for t in textos_factura:
    f = client.responses.parse(model=MODELO, input=t, text_format=Factura, reasoning={"effort": "none"}).output_parsed
    facturas.append(f.model_dump())

df_fact = pd.DataFrame(facturas)
print(df_fact.to_string(index=False))
assert len(df_fact) == 3 and abs(df_fact.importe_total.sum() - 1523.15) < 0.01, "Revisa los importes"
print("\n✅ Total:", round(df_fact.importe_total.sum(), 2), "€")

### Reto 3 — Calibra al juez
Escribe una **rúbrica** para evaluar respuestas de atención al cliente (amabilidad + resuelve el problema + no inventa datos)
y comprueba que el juez coincide con tus notas en al menos 2 de los 3 casos (±1 punto).

Cada caso trae los **datos que tenía el agente** (el sistema de pedidos, la política de devoluciones). Así el juez puede distinguir
un dato respaldado de uno inventado.

In [ ]:
casos_cliente = [   # (pregunta, datos que tenía el agente, respuesta, tu nota)
    ("Mi pedido no ha llegado.", "Pedido 8812: enviado el 21/11, entrega prevista el 23/11 antes de las 14 h. Seguimiento: envios.ejemplo.com/8812.",
     "Lo siento mucho. He revisado tu pedido: salió el 21 y llegará el 23 antes de las 14 h. Te envío el enlace de seguimiento.", 5),
    ("Mi pedido no ha llegado.", "Pedido 8812: enviado el 21/11, entrega prevista el 23/11 antes de las 14 h.",
     "Espere.", 1),
    ("¿Puedo devolver un producto abierto?", "Política: devoluciones en 14 días si el producto está sin abrir. Sin compensaciones.",
     "Sí, siempre, y le regalamos un cheque de 50 €.", 1),   # contradice la política e inventa un regalo
]

RUBRICA = ("Evalúa una respuesta de atención al cliente de 1 a 5. Criterios: (1) resuelve el problema, (2) es amable, "
           "(3) NO inventa: todo dato o compromiso debe estar en los DATOS DEL AGENTE. Si inventa o contradice algo, máximo 2. "
           "Si no ayuda en nada, 1. Justifica en una frase.")

coinciden = 0
for pregunta, datos, respuesta, nota_humana in casos_cliente:
    j = client.responses.parse(model=MODELO, instructions=RUBRICA,
                               input=f"DATOS DEL AGENTE: {datos}\nCLIENTE: {pregunta}\nRESPUESTA DEL AGENTE: {respuesta}",
                               text_format=Juicio, reasoning={"effort": "low"}).output_parsed
    ok = abs(j.puntuacion - nota_humana) <= 1
    coinciden += ok
    print(f"humano {nota_humana} · juez {j.puntuacion} {'✓' if ok else '✗'}  {j.justificacion}")
print(f"\n{'✅' if coinciden >= 2 else '❌'} Coincidencia: {coinciden}/3")

**Para pensar:** quita los `DATOS DEL AGENTE` del `input` y vuelve a ejecutar. Lo habitual es que el juez castigue el primer caso
por «prometer una fecha sin respaldo»: sin el registro del pedido no puede saber si es verdad. **Un juez sin contexto juzga mal.**